# 06 - Purchase Prediction

Goal: predict whether a browsing session ends in a purchase (`Revenue`), using
the Online Shoppers Purchasing Intention dataset.

This is a **different dataset from the retail transactions** used in notebooks
02-05 — there's no shared customer key between them. This notebook is a
self-contained classification task.

Key constraint: the target is imbalanced (~15.6% positive), confirmed in
notebook 01. Accuracy alone would be misleading here — a model that always
predicts "no purchase" gets ~84% accuracy while being useless.

**Cross-version compatibility fix:** this environment's pandas (3.0) defaults to the new `StringDtype` for text columns (`future.infer_string=True`). Anything fitted or computed here (OneHotEncoder categories, product-name lookups) would then get pickled with that dtype -- which older/different pandas installs (e.g. on the machine actually running the dashboard) cannot unpickle, raising `NotImplementedError` on load. Forcing `pd.set_option('future.infer_string', False)` up front keeps everything on plain `object` dtype, which pickles compatibly across pandas versions.


**Environment note:** `imbalanced-learn` (SMOTE) isn't installable here (no network access). Substituted a manual SMOTE (k-NN interpolation on minority class, train-fold only -- see `smote_pipeline.py`). Results are close but not identical to imblearn's SMOTE: precision/recall 0.66/0.67 here vs. 0.68/0.70 originally; ROC-AUC matches at 0.922 both times.


125 duplicate rows dropped (flagged back in notebook 01). Without an invoice or
session ID to distinguish two genuinely identical real sessions from a logging
duplicate, we treat exact-duplicate rows the same way we did for the retail data.

## Features and target

`OperatingSystems`, `Browser`, `Region`, and `TrafficType` are stored as integers
in the raw file, but they're **codes, not quantities** — OS "4" isn't twice OS "2"
in any meaningful sense. Treating them as numeric would tell a linear model there's
an ordinal relationship that doesn't exist. We one-hot encode them alongside the
genuinely categorical `Month` and `VisitorType`, rather than leaving them as raw
integers just because pandas read them that way.

`stratify=y` keeps the same ~15.6% positive rate in both splits — otherwise a
random split could accidentally leave the test set with a different imbalance
ratio than training, making metrics harder to interpret.

## Model selection

We compare a small, justified set rather than every algorithm in scikit-learn:

- **Logistic Regression** — simple linear baseline, fast, interpretable coefficients
- **Random Forest** — handles non-linear relationships and feature interactions
  without much tuning, robust to the mix of numeric and one-hot categorical features
- **Gradient Boosting** — typically the strongest of the three on tabular data,
  worth checking whether the extra complexity earns its keep

We skip KNN, SVM, and Naive Bayes here: KNN and SVM scale poorly and need careful
distance-metric tuning on high-dimensional one-hot data, and Naive Bayes' independence
assumption is a poor fit for correlated behavioral features like page counts and
durations. Including them "for completeness" wouldn't add insight.

## Handling the imbalance: three approaches, compared honestly

Baseline: recall on the minority class (actual purchases) is only ~0.41 —
the model misses more than half of real purchases while accuracy still looks
fine (~89%). This is the exact trap the imbalance warrants caution about.

`class_weight="balanced"` trades precision for recall — it catches more real
purchases (recall jumps to ~0.80) but flags more false positives (precision drops
to ~0.50). Whether that trade is worth it depends on the business cost of a missed
sale vs. a wasted marketing touch — not something the model can decide on its own.

SMOTE is applied **inside** the pipeline, so it only ever runs on the training
fold — it never synthesizes examples using information from the test set. This
matters: SMOTE before the split would leak test-set patterns into synthetic
training examples and inflate the reported metrics.

## Comparing all four

Reading this table:

- **Gradient Boosting has the best ROC-AUC** (~0.936) — the best at ranking sessions
  by purchase probability overall, even without explicit imbalance handling.
- **Random Forest + SMOTE has the best F1 on the positive class** (~0.69) — the best
  balance of precision and recall if you need a hard yes/no purchase call.
- **LogReg + class_weight=balanced has the highest recall** (~0.80) — best if missing
  a real purchase is much costlier than a false alarm (e.g. triggering a retention
  email costs little).

There's no single "best" model here — it depends on what the prediction is used
for. We move forward with **Random Forest + SMOTE** as the primary model because
it doesn't sacrifice precision as heavily as the recall-maximizing option, but this
is a judgment call worth restating, not an objective fact.

## Feature importance

`PageValues` is almost certainly the dominant feature — it's Google Analytics'
own measure of a page's average contribution to revenue, so it's somewhat
circular (a page with high PageValues is, by construction, associated with
purchases). Worth noting in any write-up: this isn't the model discovering a
subtle behavioral insight, it's picking up a feature that's already close to
a proxy for the target.

## Save the model

In [1]:
import pandas as pd
pd.set_option('future.infer_string', False)
import sys; sys.path.insert(0, '.')
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import NearestNeighbors
from smote_pipeline import SmotePipeline
from sklearn.metrics import (
    classification_report, roc_auc_score, confusion_matrix,
    ConfusionMatrixDisplay, RocCurveDisplay, PrecisionRecallDisplay
)

df = pd.read_csv("../data/raw/online_shoppers_intention.csv")
df = df.drop_duplicates()
print("shape:", df.shape)

y = df["Revenue"].astype(int)
X = df.drop(columns=["Revenue"])
numeric_cols = ["Administrative", "Administrative_Duration", "Informational", "Informational_Duration",
    "ProductRelated", "ProductRelated_Duration", "BounceRates", "ExitRates", "PageValues", "SpecialDay"]
categorical_cols = ["Month", "OperatingSystems", "Browser", "Region", "TrafficType", "VisitorType", "Weekend"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(X_train.shape, X_test.shape)
print("Train positive rate:", y_train.mean().round(4))

preprocess = ColumnTransformer([
    ("num", StandardScaler(), numeric_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols)
])

results = {}
def evaluate(name, pipe):
    pipe.fit(X_train, y_train)
    preds = pipe.predict(X_test)
    probs = pipe.predict_proba(X_test)[:, 1]
    report = classification_report(y_test, preds, output_dict=True)
    auc = roc_auc_score(y_test, probs)
    results[name] = {"precision_1": report["1"]["precision"], "recall_1": report["1"]["recall"],
                      "f1_1": report["1"]["f1-score"], "roc_auc": auc}
    print(f"--- {name} ---"); print(classification_report(y_test, preds)); print("ROC-AUC:", round(auc, 4)); print()
    return pipe

pipe_baseline = Pipeline([("prep", preprocess), ("clf", LogisticRegression(max_iter=1000, random_state=42))])
_ = evaluate("LogReg (no imbalance handling)", pipe_baseline)

pipe_balanced = Pipeline([("prep", preprocess), ("clf", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42))])
_ = evaluate("LogReg (class_weight=balanced)", pipe_balanced)

# --- Manual SMOTE, train-fold only (imblearn unavailable, no network to install it) ---
# Fit preprocessing on train only, oversample the minority class in TRANSFORMED train
# space via k-NN interpolation (SMOTE's actual mechanism), fit RF on the oversampled
# training data, then wrap preprocess+clf in a small object so .predict/.predict_proba
# work identically to a real pipeline and it saves/loads the same way with joblib.
preprocess_smote = ColumnTransformer([
    ("num", StandardScaler(), numeric_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols)
])
pipe_smote = SmotePipeline(preprocess_smote, RandomForestClassifier(n_estimators=200, random_state=42))
pipe_smote.fit(X_train, y_train)
preds = pipe_smote.predict(X_test)
probs = pipe_smote.predict_proba(X_test)[:, 1]
report = classification_report(y_test, preds, output_dict=True)
auc = roc_auc_score(y_test, probs)
results["Random Forest + SMOTE (manual)"] = {"precision_1": report["1"]["precision"], "recall_1": report["1"]["recall"],
                                               "f1_1": report["1"]["f1-score"], "roc_auc": auc}
print("--- Random Forest + SMOTE (manual) ---"); print(classification_report(y_test, preds)); print("ROC-AUC:", round(auc, 4)); print()

pipe_gb = Pipeline([("prep", preprocess), ("clf", GradientBoostingClassifier(random_state=42))])
_ = evaluate("Gradient Boosting (no imbalance handling)", pipe_gb)

comparison = pd.DataFrame(results).T.round(3)
print(comparison)

final_model = pipe_smote
preds = final_model.predict(X_test); probs = final_model.predict_proba(X_test)[:, 1]
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
cm = confusion_matrix(y_test, preds)
ConfusionMatrixDisplay(cm, display_labels=["No Purchase", "Purchase"]).plot(ax=axes[0], colorbar=False)
axes[0].set_title("Confusion Matrix")
RocCurveDisplay.from_predictions(y_test, probs, ax=axes[1]); axes[1].set_title("ROC Curve")
PrecisionRecallDisplay.from_predictions(y_test, probs, ax=axes[2]); axes[2].set_title("Precision-Recall Curve")
plt.tight_layout(); plt.savefig("../reports/figures/purchase_prediction_evaluation.png"); plt.close()

feature_names = final_model.named_steps["prep"].get_feature_names_out()
importances = final_model.named_steps["clf"].feature_importances_
importance_df = pd.DataFrame({"feature": feature_names, "importance": importances}).sort_values("importance", ascending=False).head(15)
fig, ax = plt.subplots(figsize=(9, 6))
ax.barh(importance_df["feature"][::-1], importance_df["importance"][::-1])
ax.set_title("Top 15 Feature Importances (Random Forest)")
plt.tight_layout(); plt.savefig("../reports/figures/purchase_prediction_feature_importance.png"); plt.close()
print(importance_df)

import joblib
joblib.dump(final_model, "../models/purchase_model.pkl")
print("saved")


shape: (12205, 18)
(9764, 17) (2441, 17)
Train positive rate: 0.1563
--- LogReg (no imbalance handling) ---
              precision    recall  f1-score   support

           0       0.90      0.97      0.94      2059
           1       0.75      0.41      0.53       382

    accuracy                           0.89      2441
   macro avg       0.83      0.69      0.73      2441
weighted avg       0.88      0.89      0.87      2441

ROC-AUC: 0.8996

--- LogReg (class_weight=balanced) ---
              precision    recall  f1-score   support

           0       0.96      0.86      0.90      2059
           1       0.50      0.80      0.62       382

    accuracy                           0.85      2441
   macro avg       0.73      0.83      0.76      2441
weighted avg       0.89      0.85      0.86      2441

ROC-AUC: 0.9097

--- Random Forest + SMOTE (manual) ---
              precision    recall  f1-score   support

           0       0.94      0.94      0.94      2059
           1     